# 📊 F1 Historical Features - Exploratory Data Analysis

**Goal:** Explore the ML-ready dataset with historical features

**Contents:**
1. Load and validate data
2. Dataset overview and statistics
3. Feature distributions
4. Missing data analysis
5. Correlation analysis
6. Historical feature effectiveness
7. Weather impact analysis
8. Team performance trends

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Pandas display options
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

print("✅ Imports successful")

---
## 1️⃣ Load Data

In [ ]:
# Load features
df = pd.read_parquet('../data/features/ml_features_2022_2025.parquet')

print(f"📦 Dataset shape: {df.shape}")
print(f"📅 Years: {df['year'].min()} - {df['year'].max()}")
print(f"🏎️  Drivers: {df['driver'].nunique()}")
print(f"🏁 Events: {df['event'].nunique()}")
print(f"🎯 Sessions: {df['session'].nunique()}")

df.head()

In [ ]:
# Data types
df.info()

---
## 2️⃣ Dataset Overview

In [ ]:
# Sessions per year
sessions_per_year = df.groupby(['year', 'session']).size().reset_index(name='count')

fig = px.bar(
    sessions_per_year,
    x='year',
    y='count',
    color='session',
    title='📊 Driver-Sessions per Year by Session Type',
    labels={'count': 'Driver-Sessions', 'year': 'Year'},
    height=500
)
fig.show()

In [ ]:
# Target variable distribution
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Qualifying Position Distribution', 'Race Position Distribution')
)

# Qualifying
qual_data = df['qualifying_position'].dropna()
fig.add_trace(
    go.Histogram(x=qual_data, name='Qualifying', nbinsx=20, marker_color='#3498db'),
    row=1, col=1
)

# Race
race_data = df['race_position'].dropna()
fig.add_trace(
    go.Histogram(x=race_data, name='Race', nbinsx=20, marker_color='#e74c3c'),
    row=1, col=2
)

fig.update_layout(height=400, showlegend=False, title_text='🎯 Target Variable Distributions')
fig.update_xaxes(title_text='Position', row=1, col=1)
fig.update_xaxes(title_text='Position', row=1, col=2)
fig.update_yaxes(title_text='Count', row=1, col=1)

fig.show()

print(f"Qualifying: {len(qual_data):,} samples, mean position: {qual_data.mean():.2f}")
print(f"Race: {len(race_data):,} samples, mean position: {race_data.mean():.2f}")

---
## 3️⃣ Feature Distributions

In [ ]:
# Identify feature groups
historical_features = [
    'circuit_avg_position', 'circuit_best_position', 'circuit_position_std',
    'recent_avg_position', 'recent_best_position', 'form_trend',
    'wet_dry_delta', 'team_circuit_avg_position', 'team_momentum'
]

telemetry_features = [
    'max_throttle_ratio', 'braking_events', 'brake_max_g', 'brake_avg_g',
    'drs_activations', 'degradation_slope'
]

weather_features = [
    'avg_rainfall', 'avg_track_temp', 'avg_air_temp'
]

tire_features = [
    'compound', 'tyre_age', 'is_fresh_tyre'
]

# Filter to existing columns
historical_features = [f for f in historical_features if f in df.columns]
telemetry_features = [f for f in telemetry_features if f in df.columns]
weather_features = [f for f in weather_features if f in df.columns]
tire_features = [f for f in tire_features if f in df.columns]

print(f"📍 Historical features: {len(historical_features)}")
print(f"🏎️  Telemetry features: {len(telemetry_features)}")
print(f"🌧️  Weather features: {len(weather_features)}")
print(f"🔧 Tire features: {len(tire_features)}")

In [ ]:
# Historical features distributions
if historical_features:
    fig = make_subplots(
        rows=3, cols=3,
        subplot_titles=historical_features
    )
    
    for idx, feat in enumerate(historical_features[:9]):
        row = idx // 3 + 1
        col = idx % 3 + 1
        
        data = df[feat].dropna()
        
        fig.add_trace(
            go.Histogram(x=data, name=feat, showlegend=False, nbinsx=30),
            row=row, col=col
        )
    
    fig.update_layout(height=800, title_text='📊 Historical Feature Distributions')
    fig.show()

In [ ]:
# Telemetry features distributions
if telemetry_features:
    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=telemetry_features
    )
    
    for idx, feat in enumerate(telemetry_features[:6]):
        row = idx // 3 + 1
        col = idx % 3 + 1
        
        data = df[feat].dropna()
        
        fig.add_trace(
            go.Histogram(x=data, name=feat, showlegend=False, nbinsx=30, marker_color='#2ecc71'),
            row=row, col=col
        )
    
    fig.update_layout(height=600, title_text='🏎️  Telemetry Feature Distributions')
    fig.show()

---
## 4️⃣ Missing Data Analysis

In [ ]:
# Calculate missing percentages
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).sort_values(ascending=False)
missing_df = pd.DataFrame({
    'feature': missing_pct.index,
    'missing_pct': missing_pct.values
}).query('missing_pct > 0')

# Plot
fig = px.bar(
    missing_df,
    x='missing_pct',
    y='feature',
    orientation='h',
    title='📉 Missing Data by Feature',
    labels={'missing_pct': 'Missing %', 'feature': 'Feature'},
    color='missing_pct',
    color_continuous_scale='Reds',
    height=600
)
fig.update_layout(showlegend=False)
fig.show()

print("\n📊 Missing Data Summary:")
print(missing_df.to_string(index=False))

In [ ]:
# Missing data patterns by year
if historical_features:
    missing_by_year = df.groupby('year')[historical_features].apply(
        lambda x: x.isnull().sum() / len(x) * 100
    ).T
    
    fig = px.imshow(
        missing_by_year,
        labels=dict(x="Year", y="Feature", color="Missing %"),
        title='🔥 Missing Data Heatmap by Year',
        aspect='auto',
        color_continuous_scale='RdYlGn_r',
        height=500
    )
    fig.show()

---
## 5️⃣ Correlation Analysis

In [ ]:
# Select numeric features for correlation
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Remove identifiers
exclude_cols = ['year', 'driver_number']
numeric_cols = [c for c in numeric_cols if c not in exclude_cols]

# Compute correlation with target
target_col = 'qualifying_position' if 'qualifying_position' in df.columns else 'race_position'

correlations = df[numeric_cols].corrwith(df[target_col]).sort_values(ascending=False)
correlations_df = pd.DataFrame({
    'feature': correlations.index,
    'correlation': correlations.values
}).dropna()

# Plot top correlations
top_n = 20
top_corr = correlations_df.nlargest(top_n, 'correlation', keep='all')

fig = px.bar(
    top_corr,
    x='correlation',
    y='feature',
    orientation='h',
    title=f'🎯 Top {top_n} Features Correlated with {target_col}',
    labels={'correlation': 'Correlation Coefficient', 'feature': 'Feature'},
    color='correlation',
    color_continuous_scale='RdBu_r',
    height=700
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
# Correlation matrix for historical features
if len(historical_features) > 1:
    corr_matrix = df[historical_features].corr()
    
    fig = px.imshow(
        corr_matrix,
        labels=dict(color="Correlation"),
        title='🔗 Historical Features Correlation Matrix',
        aspect='auto',
        color_continuous_scale='RdBu_r',
        zmin=-1, zmax=1,
        height=600
    )
    fig.show()

---
## 6️⃣ Historical Feature Effectiveness

In [ ]:
# How well does circuit history predict current performance?
if 'circuit_avg_position' in df.columns and target_col in df.columns:
    fig = px.scatter(
        df.dropna(subset=['circuit_avg_position', target_col]),
        x='circuit_avg_position',
        y=target_col,
        opacity=0.3,
        title='📍 Circuit History vs Current Performance',
        labels={
            'circuit_avg_position': 'Historical Avg Position at Circuit',
            target_col: 'Current Position'
        },
        trendline='ols',
        height=500
    )
    fig.show()
    
    # Compute correlation
    corr = df[['circuit_avg_position', target_col]].corr().iloc[0, 1]
    print(f"\n📊 Correlation: {corr:.3f}")
    print(f"{'Strong' if abs(corr) > 0.5 else 'Moderate' if abs(corr) > 0.3 else 'Weak'} predictive power")

In [ ]:
# Recent form effectiveness
if 'recent_avg_position' in df.columns and target_col in df.columns:
    fig = px.scatter(
        df.dropna(subset=['recent_avg_position', target_col]),
        x='recent_avg_position',
        y=target_col,
        opacity=0.3,
        title='📈 Recent Form vs Current Performance',
        labels={
            'recent_avg_position': 'Recent Average Position (last 5 races)',
            target_col: 'Current Position'
        },
        trendline='ols',
        height=500,
        color='form_trend' if 'form_trend' in df.columns else None
    )
    fig.show()
    
    corr = df[['recent_avg_position', target_col]].corr().iloc[0, 1]
    print(f"\n📊 Correlation: {corr:.3f}")

In [ ]:
# Form trend analysis
if 'form_trend' in df.columns and target_col in df.columns:
    # Categorize trend
    df_trend = df.dropna(subset=['form_trend', target_col]).copy()
    df_trend['trend_category'] = pd.cut(
        df_trend['form_trend'],
        bins=[-np.inf, -0.5, 0.5, np.inf],
        labels=['Improving', 'Stable', 'Declining']
    )
    
    fig = px.box(
        df_trend,
        x='trend_category',
        y=target_col,
        title='📊 Performance by Form Trend',
        labels={
            'trend_category': 'Recent Trend',
            target_col: 'Position'
        },
        color='trend_category',
        height=500
    )
    fig.show()
    
    print("\n📈 Average position by trend:")
    print(df_trend.groupby('trend_category')[target_col].mean().sort_values())

---
## 7️⃣ Weather Impact Analysis

In [ ]:
# Wet vs Dry performance
if 'avg_rainfall' in df.columns:
    df_weather = df.copy()
    df_weather['is_wet'] = df_weather['avg_rainfall'] > 0.1
    
    wet_sessions = df_weather['is_wet'].sum()
    dry_sessions = (~df_weather['is_wet']).sum()
    
    print(f"🌧️  Wet sessions: {wet_sessions:,} ({100*wet_sessions/len(df_weather):.1f}%)")
    print(f"☀️  Dry sessions: {dry_sessions:,} ({100*dry_sessions/len(df_weather):.1f}%)")
    
    # Distribution of rainfall
    fig = px.histogram(
        df_weather,
        x='avg_rainfall',
        nbins=50,
        title='🌧️  Rainfall Distribution',
        labels={'avg_rainfall': 'Average Rainfall (mm/h)'},
        height=400
    )
    fig.add_vline(x=0.1, line_dash="dash", line_color="red", annotation_text="Wet threshold")
    fig.show()

In [ ]:
# Drivers who excel in wet conditions
if 'wet_dry_delta' in df.columns:
    # Get average wet_dry_delta per driver
    wet_dry_by_driver = df.groupby('driver')['wet_dry_delta'].agg(['mean', 'count']).reset_index()
    wet_dry_by_driver = wet_dry_by_driver[wet_dry_by_driver['count'] >= 10]  # Min 10 sessions
    wet_dry_by_driver = wet_dry_by_driver.sort_values('mean')
    
    # Top 10 rain masters (negative delta = better in wet)
    top_wet = wet_dry_by_driver.head(10)
    
    fig = px.bar(
        top_wet,
        x='mean',
        y='driver',
        orientation='h',
        title='🌧️  Top 10 Rain Masters (Better in Wet than Dry)',
        labels={'mean': 'Wet-Dry Position Delta (negative = better in wet)', 'driver': 'Driver'},
        color='mean',
        color_continuous_scale='Blues_r',
        height=500
    )
    fig.update_layout(yaxis={'categoryorder': 'total ascending'})
    fig.show()

---
## 8️⃣ Team Performance Trends

In [ ]:
# Team performance over time
if 'team' in df.columns and target_col in df.columns:
    team_evolution = df.groupby(['year', 'team'])[target_col].mean().reset_index()
    
    fig = px.line(
        team_evolution,
        x='year',
        y=target_col,
        color='team',
        title='🏎️  Team Performance Evolution',
        labels={target_col: 'Average Position', 'year': 'Year'},
        markers=True,
        height=600
    )
    fig.update_yaxes(autorange='reversed')  # Lower position = better
    fig.show()

In [ ]:
# Team momentum correlation
if 'team_momentum' in df.columns and target_col in df.columns:
    df_momentum = df.dropna(subset=['team_momentum', target_col]).copy()
    df_momentum['momentum_category'] = pd.cut(
        df_momentum['team_momentum'],
        bins=[-np.inf, -0.3, 0.3, np.inf],
        labels=['Improving', 'Stable', 'Declining']
    )
    
    fig = px.box(
        df_momentum,
        x='momentum_category',
        y=target_col,
        title='📊 Performance by Team Momentum',
        labels={'momentum_category': 'Team Development Trend', target_col: 'Position'},
        color='momentum_category',
        height=500
    )
    fig.show()
    
    print("\n🏁 Average position by team momentum:")
    print(df_momentum.groupby('momentum_category')[target_col].mean().sort_values())

---
## 💾 Summary Statistics

In [ ]:
# Feature statistics
feature_groups = {
    'Historical': historical_features,
    'Telemetry': telemetry_features,
    'Weather': weather_features
}

for group_name, features in feature_groups.items():
    if features:
        print(f"\n{'='*60}")
        print(f"{group_name} Features Summary")
        print(f"{'='*60}")
        print(df[features].describe().T[['mean', 'std', 'min', '50%', 'max']].round(3))